In [0]:
# Create dummy data
data = [("Alice", "Sales", 50000), ("Bob", "Engineering", 75000), ("Charlie", "Sales", 60000)]
columns = ["Name", "Department", "Salary"]

# Convert to Spark DataFrame
df = spark.createDataFrame(data, columns)

# Display the results
display(df)

Databricks visualization. Run in Databricks to view.

In [0]:
# Save the dataframe as a Delta table named 'employee_salaries'
df.write.format("delta").mode("overwrite").saveAsTable("employee_salaries")
print("Table saved successfully!")

In [0]:
%sql
SELECT Department, AVG(Salary) as Average_Salary
FROM employee_salaries
GROUP BY Department;

In [0]:
# Read the raw CSV file from the uploaded location
# Note: Update the path below based on your specific upload location
raw_df = spark.read.format("csv") \
  .option("header", "true") \
  .option("inferSchema", "true") \
  .load("/Volumes/workspace/default/sales_data/salesforce_opportunities.csv") 

# Save as a Bronze Delta Table
raw_df.write.format("delta").mode("overwrite").saveAsTable("salesforce_bronze")
print("Bronze table created successfully.")

In [0]:
from pyspark.sql.functions import col

# Read from the Bronze table
bronze_df = spark.read.table("salesforce_bronze")

# Cleanse the data: drop null amounts and ensure integer typing
silver_df = bronze_df.dropna(subset=["Amount"]) \
                     .withColumn("Amount", col("Amount").cast("integer"))

# Save as a Silver Delta Table
silver_df.write.format("delta").mode("overwrite").saveAsTable("salesforce_silver")
print("Silver table created successfully.")

In [0]:
%sql
-- Aggregate total pipeline revenue by stage
CREATE OR REPLACE TABLE salesforce_gold AS
SELECT Stage, SUM(Amount) as Total_Pipeline_Value
FROM salesforce_silver
GROUP BY Stage;

-- View the final aggregated data
SELECT * FROM salesforce_gold;

Databricks visualization. Run in Databricks to view.